# 변분 오토인코더 실습

**Variational Autoencoder · VAE**

잠재 공간을 확률분포로 학습해 새로운 표본을 생성할 수 있게 만든 오토인코더.

소재 분야에서 이해하기: 잠재 공간을 이동시키며 새로운 조성 후보를 만든다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [변분 오토인코더 원논문](https://arxiv.org/abs/1312.6114)

## 1. 재매개화 트릭

VAE의 핵심 부품을 하나씩 확인합니다. 전체 학습은 딥러닝 프레임워크가 필요하므로,
여기서는 재매개화와 KL 항의 성질만 직접 계산합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

mu, log_sigma2 = 1.2, np.log(0.35 ** 2)
sigma = np.exp(0.5 * log_sigma2)

# 확률 변수에서 직접 뽑는 대신, 표준정규 잡음을 변환합니다(재매개화).
epsilon = rng.normal(0, 1, 20000)
z_reparam = mu + sigma * epsilon
z_direct = rng.normal(mu, sigma, 20000)
print('직접 표본 평균 %.3f 표준편차 %.3f' % (z_direct.mean(), z_direct.std()))
print('재매개화 표본 평균 %.3f 표준편차 %.3f' % (z_reparam.mean(), z_reparam.std()))
plt.hist(z_direct, bins=60, alpha=0.5, density=True, label='direct sampling')
plt.hist(z_reparam, bins=60, alpha=0.5, density=True, label='reparameterised')
plt.legend(); plt.xlabel('z'); plt.show()
print('\n두 분포가 같으므로 평균과 표준편차에 대한 미분이 가능해지고, 그래서 학습이 됩니다.')

## 2. KL 항이 잠재 분포를 표준정규로 당깁니다

In [ ]:
def kl_to_standard_normal(mu, sigma):
    return 0.5 * (sigma ** 2 + mu ** 2 - 1 - 2 * np.log(sigma))

for mu_value, sigma_value in [(0.0, 1.0), (0.0, 0.3), (2.0, 1.0), (2.0, 0.3)]:
    print('mu=%.1f sigma=%.1f -> KL %.3f' % (mu_value, sigma_value,
          kl_to_standard_normal(mu_value, sigma_value)))

mus = np.linspace(-3, 3, 200)
plt.plot(mus, kl_to_standard_normal(mus, 1.0), label='sigma = 1.0')
plt.plot(mus, kl_to_standard_normal(mus, 0.5), label='sigma = 0.5')
plt.xlabel('latent mean'); plt.ylabel('KL divergence'); plt.legend(); plt.show()
print('KL은 mu=0, sigma=1 에서 0이고 멀어질수록 커집니다. 재구성 오차와 이 항의 균형이 VAE의 손실입니다.')

## 3. 잠재 공간을 따라 이동하며 후보 만들기

결정론적 오토인코더의 잠재 공간을 확률적으로 뽑아 새 스펙트럼을 만들어 봅니다.

In [ ]:
from sklearn.neural_network import MLPRegressor

grid = np.linspace(0, 1, 40)
local = np.random.default_rng(0)
centre = local.uniform(0.25, 0.75, 600); width = local.uniform(0.04, 0.12, 600)
data = np.exp(-((grid[None, :] - centre[:, None]) ** 2) / (2 * width[:, None] ** 2))

encoder = MLPRegressor(hidden_layer_sizes=(16, 2, 16), max_iter=3000, random_state=0).fit(data, data)

def encode(X):
    activation = X @ encoder.coefs_[0] + encoder.intercepts_[0]
    activation = np.maximum(activation, 0)
    return activation @ encoder.coefs_[1] + encoder.intercepts_[1]

def decode(z):
    activation = np.maximum(z @ encoder.coefs_[2] + encoder.intercepts_[2], 0)
    return activation @ encoder.coefs_[3] + encoder.intercepts_[3]

latent = encode(data)
mean, cov = latent.mean(0), np.cov(latent.T)
samples = local.multivariate_normal(mean, cov, 4)
for index, sample in enumerate(samples):
    plt.plot(grid, decode(sample[None, :])[0], label='sample %d' % (index + 1))
plt.xlabel('channel'); plt.ylabel('intensity'); plt.legend(); plt.show()
print('잠재 분포에서 뽑아 복호하면 학습 데이터에 없던 스펙트럼이 나옵니다.')
print('진짜 VAE는 이 잠재 분포를 학습 과정에서 직접 규제합니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#vae)을 여세요.